# Day 24: Regression Metrics

**Dataset:** `Diabetes_Dataset.csv`

This notebook evaluates a regression model using MAE, MSE, RMSE, and R², and compares the
interpretation of each metric.

**Note on the target column:** this dataset's "natural" target, `Diabetes_Outcome`, is a
binary 0/1 label, not a continuous value. Since regression metrics need a continuous target to
be meaningful, this notebook instead predicts **`Glucose_Level`** (a real, continuous health
measurement) from the patient's other recorded health data.

## 1. Import Libraries and Load the Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
df = pd.read_csv("../datasets/Diabetes_Dataset.csv")

# Preview the data
df.head()

,Patient_ID,Age,Gender,BMI,Glucose_Level,Blood_Pressure,Insulin,Physical_Activity,Family_History,Diabetes_Outcome
0,P0001,58,F,36.4,151,102,98.5,86,Yes,1
1,P0002,71,M,28.3,185,80,294.3,29,No,0
2,P0003,48,M,27.1,172,82,186.5,108,Yes,1
3,P0004,34,F,24.0,86,139,181.0,46,No,0
4,P0005,62,M,19.2,83,106,228.2,86,Yes,0


In [2]:
print(df.shape)
df.info()

(500, 10)
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Patient_ID         500 non-null    str    
 1   Age                500 non-null    int64  
 2   Gender             500 non-null    str    
 3   BMI                500 non-null    float64
 4   Glucose_Level      500 non-null    int64  
 5   Blood_Pressure     500 non-null    int64  
 6   Insulin            500 non-null    float64
 7   Physical_Activity  500 non-null    int64  
 8   Family_History     500 non-null    str    
 9   Diabetes_Outcome   500 non-null    int64  
dtypes: float64(2), int64(5), str(3)
memory usage: 39.2 KB


## 2. Encoding Categorical Columns

`Gender` and `Family_History` are text columns and need to be encoded to numbers before a
regression model can use them. `Patient_ID` is just an identifier, so it's dropped.

In [3]:
df = df.drop(columns=["Patient_ID"])

df["Gender"] = df["Gender"].map({"M": 1, "F": 0})
df["Family_History"] = df["Family_History"].map({"Yes": 1, "No": 0})

df.head()

,Age,Gender,BMI,Glucose_Level,Blood_Pressure,Insulin,Physical_Activity,Family_History,Diabetes_Outcome
0,58,0,36.4,151,102,98.5,86,1,1
1,71,1,28.3,185,80,294.3,29,0,0
2,48,1,27.1,172,82,186.5,108,1,1
3,34,0,24.0,86,139,181.0,46,0,0
4,62,1,19.2,83,106,228.2,86,1,0


## 3. Building a Regression Model to Evaluate

In [4]:
X = df.drop(columns=["Glucose_Level"])
y = df["Glucose_Level"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model trained. First 5 predictions vs actual values:")
pd.DataFrame({"actual": y_test.values[:5], "predicted": y_pred[:5].round(1)})

Model trained. First 5 predictions vs actual values:


,actual,predicted
0,139,158.1
1,89,123.7
2,162,132.9
3,129,162.9
4,97,106.6


## 4. Mean Absolute Error (MAE)

MAE is the average of the **absolute** differences between actual and predicted values — it's
in the **same units as the target** (here, Glucose level, mg/dL), and treats every error
equally regardless of size.

In [5]:
# Manual calculation, to see exactly what MAE does
manual_mae = np.mean(np.abs(y_test - y_pred))

# sklearn's built-in version, to confirm it matches
sklearn_mae = mean_absolute_error(y_test, y_pred)

print(f"Manual MAE:  {manual_mae:.3f}")
print(f"sklearn MAE: {sklearn_mae:.3f}")

Manual MAE:  24.208
sklearn MAE: 24.208


## 5. Mean Squared Error (MSE)

MSE squares each error before averaging, which **penalizes larger errors much more heavily**
than small ones, and its units are the target's units **squared** (Glucose², which isn't
directly interpretable on its own).

In [6]:
manual_mse = np.mean((y_test - y_pred) ** 2)
sklearn_mse = mean_squared_error(y_test, y_pred)

print(f"Manual MSE:  {manual_mse:.3f}")
print(f"sklearn MSE: {sklearn_mse:.3f}")

Manual MSE:  879.094
sklearn MSE: 879.094


## 6. Root Mean Squared Error (RMSE)

RMSE is just the square root of MSE. This brings the units back to the **original target
units** (Glucose), while still penalizing large errors more heavily than MAE does.

In [7]:
manual_rmse = np.sqrt(manual_mse)
sklearn_rmse = np.sqrt(sklearn_mse)

print(f"Manual RMSE:  {manual_rmse:.3f}")
print(f"sklearn RMSE: {sklearn_rmse:.3f}")

Manual RMSE:  29.650
sklearn RMSE: 29.650


## 7. R-squared (R²)

R² measures the **proportion of variance** in the target that the model explains, on a scale
that's typically between 0 and 1 (it can go negative for a model that performs worse than
simply predicting the mean every time). Unlike MAE/MSE/RMSE, it's **unitless**, which makes it
easy to compare across different models or different targets.

In [8]:
sklearn_r2 = r2_score(y_test, y_pred)
print(f"R2: {sklearn_r2:.3f}")
print(f"This means the model explains about {sklearn_r2*100:.1f}% of the variance in Glucose_Level.")

R2: 0.310
This means the model explains about 31.0% of the variance in Glucose_Level.


## 8. Demonstrating How MSE Penalizes Large Errors More Strongly

Two small toy examples with the same average absolute error, but very differently distributed,
show why MSE/RMSE react so differently from MAE.

In [9]:
# Example A: four consistent, moderate errors
errors_consistent = np.array([5, 5, 5, 5])

# Example B: same average absolute error, but one large outlier error
errors_with_outlier = np.array([1, 1, 1, 17])

print("Example A - consistent errors:", errors_consistent)
print("  MAE:", np.mean(np.abs(errors_consistent)))
print("  RMSE:", np.sqrt(np.mean(errors_consistent ** 2)))
print()
print("Example B - one large outlier:", errors_with_outlier)
print("  MAE:", np.mean(np.abs(errors_with_outlier)))
print("  RMSE:", np.sqrt(np.mean(errors_with_outlier ** 2)))

Example A - consistent errors: [5 5 5 5]
  MAE: 5.0
  RMSE: 5.0

Example B - one large outlier: [ 1  1  1 17]
  MAE: 5.0
  RMSE: 8.54400374531753


**Observation:** Both examples have the exact same MAE (5), but Example B's RMSE is
noticeably higher than Example A's. This is because squaring the single large error (17² = 289)
dominates the average far more than the three small errors do. This is exactly what "MSE/RMSE
penalize large errors more strongly" means in practice.

## 9. Comparison Table

In [10]:
comparison_table = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Value": [sklearn_mae, sklearn_mse, sklearn_rmse, sklearn_r2],
    "Units": ["Glucose_Level (mg/dL)", "Glucose_Level squared", "Glucose_Level (mg/dL)", "Unitless (0-1 scale)"],
    "Sensitive to outliers?": ["No (treats all errors equally)", "Yes (squares errors)", "Yes (squares errors)", "Indirectly, via the squared errors it's based on"],
    "Easy to interpret directly?": ["Yes - average error size", "No - squared units", "Yes - average error size", "Yes - % of variance explained"]
})

comparison_table

,Metric,Value,Units,Sensitive to outliers?,Easy to interpret directly?
0,MAE,24.207866,Glucose_Level (mg/dL),No (treats all errors equally),Yes - average error size
1,MSE,879.093925,Glucose_Level squared,Yes (squares errors),No - squared units
2,RMSE,29.649518,Glucose_Level (mg/dL),Yes (squares errors),Yes - average error size
3,R2,0.309503,Unitless (0-1 scale),"Indirectly, via the squared errors it's based on",Yes - % of variance explained


## 10. Model Evaluation

- **MAE ≈ 24.2** — on average, the model's Glucose_Level predictions are off by about 24.2
  mg/dL.
- **RMSE ≈ 29.6** — noticeably higher than MAE, which signals that some predictions have
  larger errors than the "typical" error MAE reports, pulling RMSE upward.
- **R² ≈ 0.31** — the model explains only about 31% of the variance in Glucose_Level. The
  remaining ~69% is driven by factors not captured in this dataset (diet, time since last
  meal, medication, genetics beyond family history, etc.), which is realistic for a volatile
  health metric like blood glucose predicted from a handful of general health indicators.
- Taken together: this model has **some** real predictive signal (R² well above 0, and
  `Diabetes_Outcome` clearly correlates with Glucose_Level in the raw data), but it is far from
  a precise predictor on its own (both the error-based metrics and R² agree on that).

## 11. Explanation of the Most Appropriate Metric

For this specific problem (predicting a patient's Glucose_Level), **RMSE is the most
appropriate primary metric**, for a few reasons:

- It's in the **same units as Glucose_Level (mg/dL)**, making it directly interpretable ("the
  model is typically off by about 29.6 mg/dL"). Unlike MSE, whose squared units aren't
  meaningful on their own.
- It **penalizes large errors more heavily** than MAE does, which matters clinically since a
  prediction that's wildly off for one patient is a more serious problem than being slightly
  off for several patients, since a large glucose misestimate could meaningfully affect a
  health decision.
- **R² should still be reported alongside RMSE, never instead of it** — R² tells you what
  fraction of variance is explained, but says nothing about how large the actual prediction
  errors are in real-world units, and a model can have a deceptively reasonable-looking R²
  while still making individual errors large enough to matter in practice (or vice versa).
  Relying on R² alone, as the hint for this task points out, can hide exactly the kind of
  information RMSE and MAE make directly visible.

## Outcome

This notebook calculated and compared all four core regression metrics, namely, MAE, MSE, RMSE, and
R², on a `LinearRegression` model predicting `Glucose_Level` from the `Diabetes_Dataset.csv`
file, computing MAE, MSE, and RMSE manually with NumPy first to confirm they matched
`scikit-learn`'s built-in functions. A toy example with the same MAE but one large outlier
error directly demonstrated how MSE/RMSE penalize large errors far more than MAE does.
Building a comparison table clarified each metric's units and sensitivity to outliers, and the
final model evaluation (MAE ≈ 24.2, RMSE ≈ 29.6, R² ≈ 0.31) showed a model with real but
limited predictive power. Working through this made clear why RMSE is generally the most
useful single metric to lead with for this kind of problem (i.e., it is interpretable in the target's own
units, while still reflecting the cost of large errors), and why R² is valuable context rather
than a stand-alone answer.